In [125]:

from src.preprocessing import load_data
from src.preprocessing import window_eeg
from src.preprocessing import prep_pipeline

In [126]:
data= load_data()

In [127]:
type(data) #this is a dict: key=name and inside is a dataframe
data.keys() #dict_keys(['EEG_Eye_State_Classification'])

dict_keys(['EEG_Eye_State_Classification'])

In [128]:
df = data['EEG_Eye_State_Classification']
type(df)

pandas.core.frame.DataFrame

In [129]:
X=df.drop(['eyeDetection'], axis=1).copy()
y=df['eyeDetection'].copy()

In [130]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
from sklearn.ensemble import RandomForestClassifier
rf= RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred= rf.predict(X_test)

In [131]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred), confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1651
           1       0.94      0.90      0.92      1345

    accuracy                           0.93      2996
   macro avg       0.93      0.93      0.93      2996
weighted avg       0.93      0.93      0.93      2996
 [[1579   72]
 [ 135 1210]]


In [132]:
prep = prep_pipeline()
type(prep)

sklearn.pipeline.Pipeline

In [133]:
type(window_eeg)

function

In [134]:
X_windows, y_windows = window_eeg(df, target = 'eyeDetection') # go to preprocessing to understand
print(X_windows.shape, y_windows.shape) #these are numpys

X reshaped shape is (468, 32, 14)
(468, 448) (468,)


In [135]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
scaler = StandardScaler()
pca = PCA(n_components = .95)

X_train, X_test, y_train, y_test = train_test_split(X_windows, y_windows, test_size = 0.2, random_state = 42)
X_train_scaled =scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [136]:
import matplotlib.pyplot as plt
import numpy as np
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of components")
plt.axhline(0.95, color='red', linestyle='--')  # 95% variance line
plt.ylabel("Cumulative variance")
plt.show()

AttributeError: 'PCA' object has no attribute 'explained_variance_ratio_'

In [124]:
X_train_scaled_pca = pca.fit_transform(X_train_scaled)
X_test_scaled_pca = pca.transform(X_test_scaled)

In [35]:
print(X_train_scaled_pca.shape, X_test_scaled_pca.shape)
print(y_train.shape, y_test.shape)

(374, 28) (94, 28)
(374,) (94,)


In [36]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled_pca, y_train)
y_pred = lr.predict(X_test_scaled_pca)

In [137]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred), confusion_matrix(y_test, y_pred))

ValueError: Found input variables with inconsistent numbers of samples: [94, 2996]

In [40]:
type(X_train_scaled_pca)

numpy.ndarray

In [41]:
X = np.concatenate((X_train_scaled_pca, X_test_scaled_pca), axis=0)
print(X.shape)
y=np.concatenate((y_train, y_test), axis=0)
print(y.shape)

(468, 28)
(468,)


In [42]:
from ripser import ripser
from src.features import compute_persistence, vectorize_diagrams

In [43]:
diagrams = compute_persistence(X)

In [44]:
print(diagrams) #persistence diagrams of birth, death

[[array([[0.00000000e+00, 8.47264659e-03],
       [0.00000000e+00, 1.01646446e-02],
       [0.00000000e+00, 1.09966705e-02],
       [0.00000000e+00, 2.04461068e-02],
       [0.00000000e+00, 3.55331562e-02],
       [0.00000000e+00, 4.10043597e-02],
       [0.00000000e+00, 5.53690232e-02],
       [0.00000000e+00, 6.10217154e-02],
       [0.00000000e+00, 6.76664039e-02],
       [0.00000000e+00, 7.16256201e-02],
       [0.00000000e+00, 1.18328691e-01],
       [0.00000000e+00, 1.20665357e-01],
       [0.00000000e+00, 1.35464028e-01],
       [0.00000000e+00, 1.46248102e-01],
       [0.00000000e+00, 2.13919312e-01],
       [0.00000000e+00, 2.18352348e-01],
       [0.00000000e+00, 2.69000858e-01],
       [0.00000000e+00, 2.84712493e-01],
       [0.00000000e+00, 2.88191289e-01],
       [0.00000000e+00, 3.45197052e-01],
       [0.00000000e+00, 4.72581178e-01],
       [0.00000000e+00, 4.79661942e-01],
       [0.00000000e+00, 5.59635460e-01],
       [0.00000000e+00, 6.36938632e-01],
       [0.0000

In [45]:
type(diagrams) #list of 2D arrays

list

In [46]:
len(diagrams)


468

In [47]:
from src.features import compute_persistence, vectorize_diagrams
features =  vectorize_diagrams(diagrams)

In [48]:
print(features)

[[20.69104272  0.73896581 11.89073467 ...  0.          0.
   0.        ]
 [34.7160111   1.23985754 13.84955266 ...  0.          0.
   0.        ]
 [15.00109889  0.53575353  4.68792604 ...  0.          0.
   0.        ]
 ...
 [ 5.52502167  0.1973222   1.59349122 ...  0.          0.
   0.        ]
 [14.68937558  0.52462056  4.95386051 ...  0.          0.
   0.        ]
 [11.51302492  0.41117946  3.92472339 ...  0.          0.
   0.        ]]


In [49]:
type(features)
features.shape

(468, 9)

In [50]:
features_train, features_test, y_train, y_test = train_test_split(features, y, random_state=22, stratify=y)

In [51]:
features_train = scaler.fit_transform(features_train)
features_test = scaler.transform(features_test)

In [52]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(features_train, y_train)
y_pred = rf.predict(features_test)
print(classification_report(y_test, y_pred),
      confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.65      0.75      0.70        65
           1       0.62      0.50      0.55        52

    accuracy                           0.64       117
   macro avg       0.64      0.63      0.63       117
weighted avg       0.64      0.64      0.63       117
 [[49 16]
 [26 26]]


In [53]:
lr=LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(features_train, y_train)
y_pred = lr.predict(features_test)
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.55      0.51      0.53        65
           1       0.44      0.48      0.46        52

    accuracy                           0.50       117
   macro avg       0.49      0.49      0.49       117
weighted avg       0.50      0.50      0.50       117

[[33 32]
 [27 25]]
